# RESNET

In [1]:
import torch
import torch.nn.functional as F
from torch import nn

import math
import random
import os
import time
from datetime import datetime

import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import scipy
from PIL import Image
import pandas as pd

from typing import Sequence

## Residual Block

### Downsample

In [2]:
# class Downsample(nn.Module):
#     def __init__(self, in_dims:int, out_dims:int, stride=1):
#         """
#             Downsample module for identity block.
#         """
#         super(Downsample, self).__init__()

#         self.layers = nn.Sequential(
#             nn.Conv2d(in_channels=in_dims, out_channels=out_dims, kernel_size=1, stride=stride, bias=False),
#             nn.BatchNorm2d(out_dims),
#         )

#     def forward(self, x):
#         return self.layers(x)

def downsample(in_dims:int, out_dims:int, stride=1):
    return nn.Sequential(
        nn.Conv2d(in_channels=in_dims, out_channels=out_dims, kernel_size=1, stride=stride, bias=False),
        nn.BatchNorm2d(out_dims)
    )

### Basic

In [3]:
class BasicResidualBlock(nn.Module):

    expansion = 1
    
    def __init__(self, 
                 in_dims:int, # input channels
                 out_dims:int, # base channel width used inside the block (hidden channels)
                 stride=1, # first stride
                 downsample: nn.Module | None = None, # module transforms the shortcut to match the residual
                 ):
        """
            Implementation for bsaic block using in resnet18 and resnet34.
        """
        super(BasicResidualBlock, self).__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels=in_dims, out_channels=out_dims, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_dims),
            nn.ReLU(),

            nn.Conv2d(in_channels=out_dims, out_channels=out_dims, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_dims),
        )

        self.identity_block = (
            downsample if downsample is not None else nn.Identity()
        )

    def forward(self, x):
        residual = self.block(x)
        identity = self.identity_block(x)

        return F.relu(residual + identity) 


### Bottleneck

In [4]:
class BottleneckResidualBlock(nn.Module):

    expansion = 4

    def __init__(self, 
                 in_dims:int, # input channels
                 out_dims:int, # base channel width used inside the block (hidden channels)
                 stride:int=1, # first stride
                 downsample: nn.Module | None = None, # module transforms the shortcut to match the residual
                 ):
        """
            Implementation for bottleneck block using in resnet50+.
        """
        super(BottleneckResidualBlock, self).__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels=in_dims, out_channels=out_dims, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_dims),
            nn.ReLU(),

            nn.Conv2d(in_channels=out_dims, out_channels=out_dims, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_dims),
            nn.ReLU(),

            nn.Conv2d(in_channels=out_dims, out_channels=out_dims * self.expansion, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_dims * self.expansion),
        )

        self.identity_block = (
            downsample if downsample is not None else nn.Identity()
        )

    def forward(self, x):
        residual = self.block(x)
        identity = self.identity_block(x)
    
        return F.relu(residual + identity) 


In [5]:
downsample(64, 256)

Sequential(
  (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
)

## Resnet

In [6]:
# Replicating the Resnet paper (setting for Imagenet 1k)
class Resnet(nn.Module):
    def __init__(self, 
                 Block: type[BasicResidualBlock] | type[BottleneckResidualBlock],
                 layers: Sequence[int],
                 in_channels: int = 3, # color RGB images
                 num_classes=1000,
                 ):
        super(Resnet, self).__init__()

        self.in_dims = 64

        # ImageNet stem (7×7, 64, stride 2 + maxpool)
        # Conv -> BN -> Relu -> maxpool
        # ImageNet stem: 224x224 -> 112x112 -> 56x56.
        # New out size = abs((H - K + 2P)/S) + 1 
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.big_layers = nn.Sequential(
            # Based on paper (table 1): spatial size 56 -> 28 -> 14 -> 7
            # stride 1 (aka no size change): spatial size after self.conv1 is the same as the one in first layer
            # stride 2 (aka half size since where applied K = 3, P = 1): latter spatial size = 0.5 * former spatial size
            self._make_layer(Block, 64, layers[0],stride=1),  
            self._make_layer(Block, 128, layers[1], stride=2),
            self._make_layer(Block, 256, layers[2],stride=2),
            self._make_layer(Block, 512, layers[3], stride=2),
        )

        # feature channels = from 64 -> 256 -> 512 -> 1024 -> 2048

        self.avg_pool = nn.AdaptiveAvgPool2d((1,1))

        # classification head
        self.fc = nn.Linear(in_features=512 * Block.expansion, out_features=num_classes)

    def _make_layer(self, 
                    Block: type[BasicResidualBlock] | type[BottleneckResidualBlock], # Block used for inner layers
                    out_dims: int, # base channels
                    number_of_blocks: int,
                    stride:int = 1,
                    ):

        out_channels = out_dims * Block.expansion
        shortcut = None

        if stride != 1 or self.in_dims != out_channels:
            shortcut = downsample(self.in_dims, out_channels, stride)

        layers = [Block(in_dims=self.in_dims, out_dims=out_dims, downsample=shortcut, stride=stride)]

        self.in_dims = out_channels

        for i in range(1, number_of_blocks):
            layers.append(
                Block(in_dims=self.in_dims, out_dims=out_dims)
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1 (x) # [N, 3, 224, 224]
        x = self.big_layers(x) # [N, 2048, 7, 7] (spatial: 56 -> 56 -> 28 -> 14 -> 7, feat: 64 -> 256 -> 512 -> 1024 -> 2048)
        x = self.avg_pool(x) # [N, 2048, 1, 1]
        x = torch.flatten(x, 1) # [N, 2048]
        x = self.fc(x) # [N, num_classes]

        return x

# Training

In [7]:
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

In [8]:
layers = [3, 4, 6, 3]
input_dim = 3
output_dim = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

torch.backends.cudnn.benchmark = True  # fixed input size -> let cudnn pick the fastest kernels

set_seed()  # seed right before weight init so the run is reproducible

# resnet50 train imagenet1k
net = Resnet(Block=BottleneckResidualBlock, 
             layers=layers,  
             num_classes=output_dim,
             in_channels=input_dim).to(device)

net = net.to(memory_format=torch.channels_last)

print(f"{sum(p.numel() for p in net.parameters()) / 1e6:.2f}M parameters")

cuda NVIDIA GeForce RTX 5080
25.56M parameters


In [9]:
for name, module in net.named_modules():
    if isinstance(module, (nn.Conv2d, nn.MaxPool2d, nn.AvgPool2d)):
        stride = module.stride
        if stride is None:
            continue

        strides = (stride,) if isinstance(stride, int) else stride
        if any(s <= 0 for s in strides):
            print(f"Invalid stride in {name}: {module}")

## Load data

In [10]:
from __future__ import annotations

import os
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
from collections import defaultdict

In [11]:
IMAGENET_DIR = './dataset/Imagenet_1k_extract'
TRAIN_DIR = os.path.join(IMAGENET_DIR, 'train')
VAL_DIR   = os.path.join(IMAGENET_DIR, 'val')

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

In [12]:
BATCH_SIZE = 128
IMAGE_SIZE = 224

train_transform = v2.Compose([
    v2.ToImage(),

    # Randomly crop and resize an ImageNet image to 224 × 224.
    v2.RandomResizedCrop(
        size=(IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.08, 1.0),
        ratio=(3 / 4, 4 / 3),
        antialias=True,
    ),

    v2.RandomHorizontalFlip(p=0.5),

    # Convert uint8 [0, 255] to float32 [0, 1].
    v2.ToDtype(torch.float32, scale=True),

    v2.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])

val_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop((IMAGE_SIZE, IMAGE_SIZE)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])

In [13]:
class NumericImageFolder(ImageFolder):
    """ImageFolder that interprets numeric folder names numerically."""

    def find_classes(
        self,
        directory: str | Path,
    ) -> tuple[list[str], dict[str, int]]:
        classes = [
            entry.name
            for entry in os.scandir(directory)
            if entry.is_dir()
        ]

        if not classes:
            raise FileNotFoundError(
                f"No class directories found in {directory}"
            )

        try:
            classes = sorted(classes, key=int)
        except ValueError as exc:
            raise ValueError(
                "NumericImageFolder requires class directories named "
                "'0', '1', ..., '999'."
            ) from exc

        class_to_idx = {
            class_name: int(class_name)
            for class_name in classes
        }

        expected_classes = list(range(len(classes)))
        actual_classes = [int(name) for name in classes]

        if actual_classes != expected_classes:
            raise ValueError(
                "Class directories must form a continuous numeric range. "
                f"Expected 0..{len(classes) - 1}."
            )

        return classes, class_to_idx

In [14]:
train_dataset = NumericImageFolder(
    root=TRAIN_DIR,
    transform=train_transform,
)

val_dataset = NumericImageFolder(
    root=VAL_DIR,
    transform=val_transform,
)

eval_train_dataset = NumericImageFolder(
    root=TRAIN_DIR,
    transform=val_transform,
)

In [15]:
def make_balanced_subset(
    dataset: NumericImageFolder,
    samples_per_class: int = 10,
    seed: int = 42,
) -> Subset:
    """
    Create a reproducible subset containing the same number of
    samples from every class.

    For ImageNet-1K:
        10 samples/class × 1,000 classes = 10,000 samples.
    """
    indices_by_class: dict[int, list[int]] = defaultdict(list)

    for sample_index, target in enumerate(dataset.targets):
        indices_by_class[target].append(sample_index)

    generator = torch.Generator().manual_seed(seed)
    selected_indices: list[int] = []

    for class_index in range(len(dataset.classes)):
        class_indices = indices_by_class[class_index]

        if len(class_indices) < samples_per_class:
            raise ValueError(
                f"Class {class_index} only contains "
                f"{len(class_indices)} samples, but "
                f"{samples_per_class} were requested."
            )

        permutation = torch.randperm(
            len(class_indices),
            generator=generator,
        )

        selected_indices.extend(
            class_indices[index]
            for index in permutation[:samples_per_class].tolist()
        )

    return Subset(
        dataset=dataset,
        indices=selected_indices,
    )


eval_train_subset = make_balanced_subset(
    dataset=eval_train_dataset,
    samples_per_class=10,
    seed=SEED,
)

In [16]:
if train_dataset.class_to_idx != val_dataset.class_to_idx:
    raise ValueError(
        "Training and validation class mappings do not match."
    )

if train_dataset.class_to_idx != eval_train_dataset.class_to_idx:
    raise ValueError(
        "Training and evaluation class mappings do not match."
    )

print(f"Training images:          {len(train_dataset):,}")
print(f"Validation images:        {len(val_dataset):,}")
print(f"Evaluation subset images: {len(eval_train_subset):,}")
print(f"Number of classes:        {len(train_dataset.classes):,}")

Training images:          1,281,167
Validation images:        50,000
Evaluation subset images: 10,000
Number of classes:        1,000


## Data loader

In [17]:
NUM_WORKERS = min(8, os.cpu_count() or 1)
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=False,
)

eval_train_loader = DataLoader(
    dataset=eval_train_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

In [18]:
images, targets = next(iter(train_loader))

print("Image batch:", images.shape)
print("Target batch:", targets.shape)
print("Image dtype:", images.dtype)
print("Target dtype:", targets.dtype)
print("Target range:", targets.min().item(), targets.max().item())

Image batch: torch.Size([128, 3, 224, 224])
Target batch: torch.Size([128])
Image dtype: torch.float32
Target dtype: torch.int64
Target range: 3 996


In [19]:
images = images.to(
    device,
    non_blocking=True,
)

if device.type == "cuda":
    images = images.contiguous(
        memory_format=torch.channels_last
    )

targets = targets.to(
    device,
    non_blocking=True,
)

net.eval()

with torch.inference_mode():
    logits = net(images)

print("Output shape (batch_size, num_classes):", logits.shape)

assert logits.shape == (
    images.size(0),
    output_dim,
)

Output shape (batch_size, num_classes): torch.Size([128, 1000])


# Run

In [20]:
epochs = 100

# Label smoothing caps logit over-confidence; the printed loss will floor
# near ~0.5 instead of 0 — that is expected, not a bug.
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Weight decay goes on conv/linear weights ONLY: decaying BN affine params and
# biases (everything with ndim == 1) fights the normalization instead of regularizing.
decay, no_decay = [], []
for name, p in net.named_parameters():
    (no_decay if p.ndim == 1 else decay).append(p)

optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)

# 5-epoch linear warmup (0.01 -> 0.1) guards against early divergence at lr=0.1,
# then cosine decay to 0. Stepped once per epoch in run().
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - 5, eta_min=0.0)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

In [21]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0

    model.eval()

    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                scores = model(x)
            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum().item()
            num_samples += predictions.size(0)

    return num_correct / num_samples

In [22]:
def run(model, optimizer, scheduler, criterion, epochs, writer, run_name):
    # AMP: forward in reduced precision, gradients rescaled to avoid underflow.
    # enabled= flags keep the whole loop runnable on CPU too.
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    os.makedirs("checkpoints", exist_ok=True)
    best_acc = 0.0
    step = 0

    try:
        for epoch in range(epochs):
            t0 = time.perf_counter()
            running_loss = 0.0
            model.train()

            for data, targets in train_loader:
                data = data.to(device, non_blocking=True, memory_format=torch.channels_last)
                targets = targets.to(device, non_blocking=True)

                # Forward propagation
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    scores = model(data)
                    loss = criterion(scores, targets)

                # Backward propagation
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item()

                if step % 50 == 0:
                    writer.add_scalar("Loss/train_batch", loss.item(), step)
                step += 1

            avg_epoch_loss = running_loss / len(train_loader)

            # Train accuracy on the clean fixed subset, test on the full test set.
            train_acc = check_accuracy(eval_train_loader, model)
            test_acc = check_accuracy(val_loader, model)

            # Step the scheduler once per epoch
            scheduler.step()

            # Log the learning rate used for the next epoch
            current_lr = optimizer.param_groups[0]["lr"]

            epoch_time = time.perf_counter() - t0
            writer.add_scalar("Loss/train_epoch", avg_epoch_loss, epoch)
            writer.add_scalar("Accuracy/train", train_acc, epoch)
            writer.add_scalar("Accuracy/test", test_acc, epoch)
            writer.add_scalar("Accuracy/gap", train_acc - test_acc, epoch)
            writer.add_scalar("LR", current_lr, epoch)
            writer.add_scalar("Time/epoch_sec", epoch_time, epoch)

            # Keep the best weights — cosine-to-zero means the last epoch is
            # usually the best, but this run is insurance against surprises.
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save({
                    "epoch": epoch,
                    "test_acc": test_acc,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                }, f"checkpoints/{run_name}_best.pt")

            print(
                f"Epoch [{epoch + 1}/{epochs}], "
                f"Loss: {avg_epoch_loss:.4f}, "
                f"Train Acc: {train_acc:.4f}, "
                f"Test Acc: {test_acc:.4f}, "
                f"LR: {current_lr:.6f}, "
                f"{epoch_time:.1f}s"
            )
    finally:
        torch.save(model.state_dict(), f"checkpoints/{run_name}_final.pt")
        writer.close()

    print(f"Best test acc: {best_acc:.4f} (checkpoints/{run_name}_best.pt)")

In [ ]:
import json
from torch.utils.tensorboard import SummaryWriter

# One directory per run so TensorBoard curves never overlap between runs.
run_name = f"{datetime.now():%Y%m%d-%H%M%S}_resnet50"
writer = SummaryWriter(f"runs/imagenet/{run_name}")
writer.add_text("config", json.dumps({
    "arch": "resnet50-imagenet1k", "layers": layers,
    "optimizer": "AdamW", "lr": 1e-3, "weight_decay": 1e-2, 
    "scheduler": "warmup5+cosine", "epochs": epochs, "batch_size": BATCH_SIZE,
    "label_smoothing": 0.1, "augmentation": "crop-reflect+hflip+erasing", "seed": SEED,
}))

run(net, optimizer=optimizer, scheduler=scheduler, criterion=criterion,
    epochs=epochs, writer=writer, run_name=run_name)

### ImageNet-1K — Fine-tuning Result

#### ResNet-50 v1.0 — full ImageNet-1K training run (2026-09-22)

Architecture + hyperparameters (**25.56M params**):

    * Flow (paper resnet-50, table 1) = input(3,224,224)
      -> stem: conv(64,7,2,3) -> BN -> ReLU -> maxpool(3,2,1)                  # 224 -> 56
      -> layer1: 3 x Bottleneck(64,  expansion 4), stride 1                    # 56 -> 56, C 64->256
      -> layer2: 4 x Bottleneck(128, expansion 4), stride 2 on block 1         # 56 -> 28, C 256->512
      -> layer3: 6 x Bottleneck(256, expansion 4), stride 2 on block 1         # 28 -> 14, C 512->1024
      -> layer4: 3 x Bottleneck(512, expansion 4), stride 2 on block 1         # 14 -> 7,  C 1024->2048
      -> AdaptiveAvgPool2d(1,1) -> Flatten -> Linear(2048, 1000)
      Bottleneck = conv(1x1) -> BN -> ReLU -> conv(3x3, stride) -> BN -> ReLU -> conv(1x1, x4) -> BN,
      shortcut = downsample(conv 1x1 + BN) when stride != 1 or channels change
    * Loss = CrossEntropyLoss(label_smoothing=0.1) -> loss floors well above 0, expected
    * Optimiser = AdamW, lr=1e-3, weight_decay=1e-2 — a `decay`/`no_decay` (ndim==1) param
      split is built in the cell but **not** passed to the optimiser; `net.parameters()` is
      used directly, so weight decay is applied to BN/bias params too (dead code, not an
      active safeguard)
    * Schedule = 5-epoch linear warmup (start_factor 0.1) -> cosine to 0 over the remaining
      95 epochs; 100 epochs total, batch_size 128, image_size 224
    * Data = train: RandomResizedCrop(224, scale=(0.08,1.0), ratio=(3/4,4/3)) + HFlip ->
      Normalize(ImageNet mean/std); val: Resize(256) -> CenterCrop(224) -> Normalize.
      The TensorBoard config text logs `"augmentation": "crop-reflect+hflip+erasing"`, but
      that string is stale/copied from the CIFAR runs — there is no reflect-pad crop and no
      RandomErasing anywhere in this pipeline.
    * Dataset = full ImageNet-1K extract at `./dataset/Imagenet_1k_extract`: 1,281,167 train
      images, 50,000 val images, 1,000 classes (`NumericImageFolder`, class dirs `"0".."999"`)
    * Systems = AMP + channels_last + cudnn.benchmark; seeded (42)
    * Eval = train accuracy on a clean, balanced fixed subset (10 images/class = 10,000,
      no augmentation, val_transform); test accuracy on the full 50,000-image val set

Result (verified run `20260922-145555_resnet50`, RTX 5080, ~881-893s/epoch, ~24.5 GPU-hours
total):

    Best/final (epoch 100): Loss 1.6377, Train Acc 0.9615, Test Acc 0.7661, LR -> 0.0
    Train/test gap = 19.5 points

Trajectory:

    | epoch | loss   | train acc | test acc | LR       |
    |-------|--------|-----------|----------|----------|
    | 1     | 5.2870 | 0.2562    | 0.2401   | 0.000280 |
    | 10    | 2.6566 | 0.7037    | 0.6427   | 0.000993 |
    | 20    | 2.4130 | 0.7586    | 0.6826   | 0.000940 |
    | 30    | 2.3052 | 0.7896    | 0.6925   | 0.000839 |
    | 40    | 2.2157 | 0.8111    | 0.7093   | 0.000701 |
    | 50    | 2.1203 | 0.8381    | 0.7217   | 0.000541 |
    | 60    | 2.0123 | 0.8656    | 0.7314   | 0.000377 |
    | 70    | 1.8902 | 0.9035    | 0.7488   | 0.000227 |
    | 80    | 1.7655 | 0.9367    | 0.7563   | 0.000105 |
    | 90    | 1.6721 | 0.9566    | 0.7643   | 0.000027 |
    | 100   | 1.6377 | 0.9615    | 0.7661   | 0.000000 |  <- best

Read:

    * **76.61% top-1 on the real 50k-image ImageNet-1K val set** — in the same band as the
      original ResNet-50 paper (~75.3% top-1, 90-epoch SGD recipe) despite using AdamW instead
      of SGD+momentum and 100 vs 90 epochs. That's a strong signal the bottleneck block, stem,
      and downsample shortcuts are implemented correctly at full ImageNet scale, not just on
      a toy subset.
    * The 19.5-point train/test gap looks large but is not straightforwardly "overfitting":
      train accuracy is measured on a fixed 10k-image subset *drawn from* the 1.28M-image
      train set and evaluated without augmentation, after 100 epochs — the same clean-eval
      methodology used in the CIFAR/Oxford runs, which inflates the apparent gap versus a
      true held-out comparison.
    * Two logging/config issues carried over from earlier notebooks, same pattern as the
      CIFAR-10 v3.0 ViT run: (1) the decay/no_decay optimiser-param split is dead code, and
      (2) the config text's augmentation string doesn't match the actual transform pipeline.
      Neither affects the result above, but both should be fixed before trusting the logged
      config on a future run.
    * Per-epoch time is flat (~881-893s) across all 100 epochs — no data-loading slowdown or
      thermal throttling over the ~24.5-hour run.
